# 01 - Data Exploration

First look at the credit card fraud dataset (Kaggle: mlg-ulb/creditcardfraud).

## Setup

Add the project root to the path so we can import from `src`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.data import load_data

## 1. Loading the data

Load the raw CSV with `load_data()`, which also checks the expected columns.

In [ ]:
df = load_data()

### Shape

Number of rows (transactions) and columns (features + target).

In [ ]:
df.shape

### Data types

Type of each column, to confirm everything is numeric as expected.

In [ ]:
df.dtypes

### First rows

A quick look at the raw values.

In [ ]:
df.head()

### Summary statistics

Count, mean, std, min, quartiles and max for each column.

In [ ]:
df.describe()

### Missing values

Number of null values per column.

In [ ]:
df.isnull().sum()

### Duplicate rows

Number of rows that are exact copies of another row.

In [ ]:
df.duplicated().sum()

## 2. Class imbalance

How many transactions are legitimate (`Class = 0`) versus fraud (`Class = 1`).

In [ ]:
import pandas as pd

counts = df["Class"].value_counts().sort_index()
percent = df["Class"].value_counts(normalize=True).sort_index() * 100

pd.DataFrame({"count": counts, "percent": percent.round(3)})

### Class distribution (log scale)

A log scale is used because the fraud bar would be invisible on a linear scale.

In [ ]:
import matplotlib.pyplot as plt

labels = ["Legitimate (0)", "Fraud (1)"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, counts.values, color=["#8c8c8c", "#d1495b"], width=0.5)
ax.set_yscale("log")
ax.set_ylim(bottom=1)
ax.set_ylabel("Number of transactions (log scale)")
ax.set_title("Class distribution")
ax.bar_label(bars, labels=[f"{c:,}" for c in counts.values], padding=3)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Why accuracy is misleading here

Only about 0.17% of transactions are fraud. A model that **always predicts "legitimate"** would reach about **99.8% accuracy** while catching **zero** frauds.

So accuracy says almost nothing about how well we detect fraud. Metrics that focus on the minority class are more useful, such as precision, recall, F1 and the area under the precision-recall curve (PR-AUC).

## 3. Amount and Time

How the transaction amount and the time of the transaction differ between classes.

### Distribution of log1p(Amount) by class

`Amount` is very skewed, so we plot `log1p(Amount)` (log of 1 + amount). Each class is shown as a density, so the very different class sizes can be compared.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(7, 4))
for value, label, color in [(0, "Legitimate (0)", "#8c8c8c"), (1, "Fraud (1)", "#d1495b")]:
    amounts = np.log1p(df.loc[df["Class"] == value, "Amount"])
    ax.hist(amounts, bins=50, density=True, alpha=0.6, color=color, label=label)
ax.set_xlabel("log1p(Amount)")
ax.set_ylabel("Density")
ax.set_title("Distribution of log1p(Amount) by class")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### Amount statistics by class

In [ ]:
df.groupby("Class")["Amount"].describe()

### Fraud rate by hour

`Time` is the number of seconds since the first transaction in the dataset, so `hour = (Time // 3600) % 24` is an hour relative to that start, not necessarily the real clock time.

In [ ]:
df["hour"] = (df["Time"] // 3600 % 24).astype(int)
hourly = df.groupby("hour")["Class"].agg(transactions="count", frauds="sum", fraud_rate="mean")
hourly["fraud_rate"] = hourly["fraud_rate"] * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(hourly.index, hourly["fraud_rate"], color="#d1495b", width=0.8)
ax.axhline(df["Class"].mean() * 100, color="#555555", linestyle="--", linewidth=1, label="Overall fraud rate")
ax.set_xticks(range(24))
ax.set_xlabel("Hour (relative to first transaction)")
ax.set_ylabel("Fraud rate (%)")
ax.set_title("Fraud rate by hour")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

hourly

### What we observe

- **Amount:** fraud amounts have a lower median (9.25 vs 22.00) but a higher mean (122.21 vs 88.29), and a smaller maximum (2,125.87 vs 25,691.16). Both classes overlap over most of the range.
- In the histogram, fraud shows a few sharp peaks at small values (log1p(Amount) near 0 and near 0.7, i.e. amounts around 0 and 1) and one near 4.7 (around 100). With only 492 frauds, these peaks are based on few transactions.
- **Hour:** the fraud rate is clearly higher in hours 2 and 4 (about 1.7% and 1.0%) than the overall rate (~0.17%). These hours also have far fewer transactions (around 2,200-3,500 vs ~16,000 in busier hours), so the rates are based on small counts.
- The dataset covers only two days and the hour is relative to the first transaction, so this does not prove a real time-of-day pattern.

## 4. Feature relationships

Which features are most related to `Class`, and how their values differ between classes.

### Top 10 features by absolute correlation with Class

Pearson correlation between each original feature and `Class`. The helper column `hour` is left out. Bars show the absolute value; the sign is in the table.

In [ ]:
feature_cols = [col for col in df.columns if col not in ["Class", "hour"]]
correlations = df[feature_cols].corrwith(df["Class"])
top10 = correlations.reindex(correlations.abs().sort_values(ascending=False).index).head(10)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top10.index[::-1], top10.abs().values[::-1], color="#d1495b")
ax.set_xlabel("Absolute correlation with Class")
ax.set_title("Top 10 features by absolute correlation with Class")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

top10.round(3).to_frame("correlation")

### Boxplots of the top 3 features by class

Outliers are hidden to keep the boxes readable.

In [ ]:
top3 = top10.index[:3]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, feature in zip(axes, top3):
    data = [df.loc[df["Class"] == 0, feature], df.loc[df["Class"] == 1, feature]]
    ax.boxplot(data, tick_labels=["Legitimate (0)", "Fraud (1)"], showfliers=False)
    ax.set_title(feature)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Takeaways

- **Use precision-recall metrics, not accuracy.** Fraud is only ~0.17% of the data, so we will focus on PR-AUC, precision, recall and F1.
- **Use a stratified train/test split** so both sets keep the same small share of fraud.
- **Handle the class imbalance**, for example with class weights in the Random Forest, and tune the decision threshold instead of using 0.5 by default.
- **Apply `log1p` to `Amount`.** It is very skewed; this matters little for a Random Forest but keeps the feature easier to inspect and compare.
- **V17, V14 and V12 separate the classes clearly** (fraud values are much lower). Keep all features for now, and check feature importance after training. Also decide what to do with the 1,081 duplicate rows before splitting.